# 项目一：电信客户流失预测（Telco Customer Churn）

**这个项目要回答的业务问题：** 哪些客户最可能流失（Churn）？流失和什么因素有关？如果只能挽留一小部分客户，应该优先挽留谁？

**为什么做这个项目：** 数据量小（7043 行）、结构干净，能在一周内跑通完整流程：拿数据 → 清洗 → 探索 → 建模 → 评估 → 写结论。而且逻辑回归的可解释性正好发挥统计功底，是简历和面试里都经得起深挖的项目。

**数据从哪来（二选一）：**
1. Kaggle 搜 `Telco Customer Churn`，下载 `WA_Fn-UseC_-Telco-Customer-Churn.csv`
2. 和鲸 HeyWhale 搜「电信客户流失」，把数据集挂载到项目里

下载后把 csv 放到和这个 Notebook 同一个目录（或改下面第 1 步里的路径）。

**最终要产出的东西（缺一不可）：**
- [ ] 一张流失率总览（整体流失率是多少）
- [ ] 至少 3 张探索图（按合约类型 / 在网时长 / 月费看流失差异）
- [ ] 两个模型的对比表（逻辑回归 vs 随机森林，5 个指标）
- [ ] 一张逻辑回归系数图（哪些因素最影响流失）
- [ ] 一段你自己写的结论 + 业务建议（在最后一步，用自己的话写）

> ⚠️ 里面标了 `# TODO(你来写)` 的地方不要跳过——面试官深挖的就是这些地方，代码可以参考，结论必须自己写。

## 第 0 步：导入工具包

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # 让中文图题不乱码
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

print('工具包导入成功')

## 第 1 步：读数据，先看它长什么样

先别急着建模。面试官最爱问的一句是"介绍一下你的数据"——这一步就是在准备这个答案：多少行、多少列、每列是什么、目标变量是哪一列。

In [ ]:
# 如果你的 csv 不在同一目录，把这里改成你的文件路径
FILE = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

try:
    df = pd.read_csv(FILE)
except FileNotFoundError:
    raise FileNotFoundError(
        f'没找到 {FILE}。请先按 Notebook 开头说明，从 Kaggle 或和鲸下载数据集，'
        '把 csv 放到和 Notebook 同一个目录再运行。'
    )

print('数据形状（行, 列）：', df.shape)
df.head()

In [ ]:
df.info()
print()
print('目标变量 Churn 的分布：')
print(df['Churn'].value_counts(normalize=True).round(3))
# TODO(你来写)：用一句话记下整体流失率是多少，写在下面的结论草稿里（最后一步要用）

## 第 2 步：数据清洗

这个数据集有一个经典的坑：`TotalCharges`（总费用）里混着空格，它会被读成字符串。先把它转成数字，转不过去的会变成缺失值——想一想：什么样的客户总费用会是空的？（答案和在网时长 tenure = 0 有关，新客户还没产生费用。）

In [ ]:
# TotalCharges 转数字，空格会变成 NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('TotalCharges 缺失数量：', df['TotalCharges'].isna().sum())

# 缺失行数很少，且都是 tenure=0 的新客户，这里选择删掉；删之前先确认一下
print(df[df['TotalCharges'].isna()]['tenure'].unique())
df = df.dropna(subset=['TotalCharges'])

# customerID 是编号，对分析没用，删掉
df = df.drop(columns=['customerID'])

# 目标变量转成 0/1：Yes(流失)=1, No=0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print('清洗后形状：', df.shape)

## 第 3 步：探索分析（EDA）——先有发现，再谈模型

每个图下面都留了 TODO。不要只画图，要写出"图告诉了我什么"。面试时被问到任何一个因素，你都要能说出它和流失的关系。

In [ ]:
# 图1：按合约类型看流失率（通常差异非常大）
contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
contract_churn.plot(kind='bar')
plt.title('Churn rate by contract type')
plt.ylabel('churn rate')
plt.xticks(rotation=0)
plt.show()
print(contract_churn.round(3))
# TODO(你来写)：月付合约(month-to-month)的流失率大约是两年合约的几倍？

In [ ]:
# 图2：流失 vs 未流失客户的在网时长分布
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True)
plt.title('Tenure distribution by churn')
plt.show()
# TODO(你来写)：流失主要集中在入网多久的客户身上？

In [ ]:
# 图3：月费 MonthlyCharges 与流失的关系
sns.boxplot(data=df, x='Churn', y='MonthlyCharges')
plt.title('Monthly charges by churn (0=stayed, 1=churned)')
plt.show()
# TODO(你来写)：再挑一个你感兴趣的变量自己画一张图（比如 InternetService 或 PaymentMethod），
# 并写一句话说清它和流失的关系。

## 第 4 步：准备建模数据

- 类别变量做独热编码（One-Hot），不然模型读不懂文字
- 按 80/20 分训练集和测试集，`stratify=y` 保证两边流失比例一致
- 数值变量做标准化——逻辑回归需要，随机森林其实不需要，但统一处理方便对比。想一想为什么标准化会影响逻辑回归的系数大小？

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = pd.get_dummies(df.drop(columns=['Churn']), drop_first=True)
y = df['Churn']
print('建模特征数：', X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('训练集：', X_train.shape, ' 测试集：', X_test.shape)

## 第 5 步：两个模型对比

逻辑回归是主力（可解释、是统计的语言），随机森林当对照组。
注意：流失客户只占约 1/4，只看 Accuracy 会骗人——一个"全猜不流失"的模型都有 73% 准确率。所以必须同时看 Precision / Recall / F1 / ROC-AUC，并想清楚业务上更怕哪种错：漏掉一个要流失的客户（Recall 低）vs 打扰一个不会流失的客户（Precision 低）。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate(model, X_tr, X_te):
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, proba),
    }, model

lr_result, lr = evaluate(LogisticRegression(max_iter=1000), X_train_scaled, X_test_scaled)
rf_result, rf = evaluate(RandomForestClassifier(n_estimators=300, random_state=42), X_train, X_test)

results = pd.DataFrame({'逻辑回归': lr_result, '随机森林': rf_result}).round(3)
results
# TODO(你来写)：哪个模型更好？如果业务目标是"尽量不错过要流失的客户"，你选哪个？为什么？

In [ ]:
# ROC 曲线（两个模型画在一张图上对比）
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots()
RocCurveDisplay.from_estimator(lr, X_test_scaled, y_test, ax=ax, name='Logistic Regression')
RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=ax, name='Random Forest')
plt.title('ROC curves')
plt.show()

## 第 6 步：解释模型——这个项目最值钱的部分

逻辑回归的系数能直接说清"哪个因素让流失概率上升/下降"。这一步产出的图和结论，是你和只会调包的人拉开差距的地方，也是文书里能写的素材。

In [ ]:
coef = pd.Series(lr.coef_[0], index=X.columns).sort_values()
plt.figure(figsize=(8, 10))
coef.plot(kind='barh')
plt.title('Logistic regression coefficients (positive = higher churn risk)')
plt.show()
print('最增加流失风险的 5 个因素：')
print(coef.tail(5).round(2))
print()
print('最降低流失风险的 5 个因素：')
print(coef.head(5).round(2))
# TODO(你来写)：用业务语言翻译前 3 个因素，不要出现"系数"两个字。
# 例：'签两年合约的客户流失风险明显更低，因为……'

## 第 7 步：写结论（必须自己写，这是面试的弹药库）

把下面模板里的括号填掉，删掉提示，就是你的一页纸项目总结。写完后存一份 PDF，和 Notebook 一起放进 GitHub。

**模板：**
- 背景：电信行业获客成本高，本项目用 7043 名客户的数据预测流失，整体流失率约为（ ）。
- 方法：清洗后将类别变量独热编码，80/20 分层划分训练/测试集；对比逻辑回归与随机森林，用 Recall 和 ROC-AUC 为主要指标（因为漏检流失客户的代价更高）。
- 结果：逻辑回归 ROC-AUC =（ ），Recall =（ ）；最重要的三个流失因素是（ ）、（ ）、（ ）。
- 业务建议：（例）对月付合约 + 入网 6 个月内 + 用光纤的客户做优先挽留触达；用合约升级优惠把月付客户转成年付。
- 不足与改进：（例）数据没有时间维度，没法验证流失是否季节性；下一步可试阈值调整提升 Recall，或加入客户行为时序特征。

**简历上的一行（按这个格式改成你的真实数字）：**
> 电信客户流失预测：基于 7043 条客户数据，构建逻辑回归与随机森林模型（ROC-AUC 0.XX），识别合约类型、在网时长等关键流失因素，提出分层挽留策略。

**面试深挖自查（答不上来的回去补）：**
- [ ] 为什么不能只看 Accuracy？
- [ ] 逻辑回归系数为正/负分别说明什么？为什么建模前要标准化？
- [ ] Precision 和 Recall 在这个业务里分别对应什么代价？你为什么更看重其中一个？
- [ ] 类别不平衡你处理了吗？除了换指标，还有什么办法（如调阈值、SMOTE）？
- [ ] 如果让你重新做一次，你会改哪一步？